[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balloontip/deep-learning/blob/main/chapter-09/09-06-GAN-MNIST-Handwritten-Digit-Generation.ipynb)

Build and train a simple Generative Adversarial Network in PyTorch to generate handwritten digits from the MNIST dataset. This project covers the Generator and Discriminator architectures, adversarial training with Binary Cross-Entropy loss, loss tracking, and visualizing how generated digits improve over time using fixed latent vectors.

In [1]:
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.utils import save_image

# -------------------------------------------------------
# Problem:
# Build a GAN that generates handwritten digits similar
# to those in the MNIST dataset.
# -------------------------------------------------------

# ----------------------------
# Hyperparameters
# ----------------------------
latent_dim = 100
image_size = 784          # 28 × 28 flattened image
batch_size = 64
num_epochs = 50
learning_rate = 0.0002

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ----------------------------
# Data Preparation
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

mnist_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

data_loader = DataLoader(
    mnist_dataset,
    batch_size=batch_size,
    shuffle=True
)

# ----------------------------
# Generator
# ----------------------------
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, image_size),
            nn.Tanh()
        )

    def forward(self, z):
        image = self.model(z)
        return image.view(image.size(0), 1, 28, 28)

# ----------------------------
# Discriminator
# ----------------------------
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(image_size, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, image):
        image = image.view(image.size(0), -1)
        return self.model(image)

# ----------------------------
# Models
# ----------------------------
generator = Generator().to(device)
discriminator = Discriminator().to(device)

loss_fn = nn.BCELoss()

# Using betas=(0.5, 0.999) is a common recommendation for
# stabilizing GAN training.
optimizer_g = torch.optim.Adam(
    generator.parameters(),
    lr=learning_rate,
    betas=(0.5, 0.999)
)

optimizer_d = torch.optim.Adam(
    discriminator.parameters(),
    lr=learning_rate,
    betas=(0.5, 0.999)
)

os.makedirs("generated_images", exist_ok=True)

# Fixed latent vectors used to visualize the Generator's
# progress consistently throughout training.
fixed_noise = torch.randn(16, latent_dim).to(device)

# ----------------------------
# Training
# ----------------------------
print("Starting training...")

for epoch in range(num_epochs):

    for real_images, _ in data_loader:

        real_images = real_images.view(real_images.size(0), -1).to(device)

        current_batch_size = real_images.size(0)

        real_labels = torch.ones(current_batch_size, 1).to(device)
        fake_labels = torch.zeros(current_batch_size, 1).to(device)

        # ----------------------------
        # Train Discriminator
        # ----------------------------
        optimizer_d.zero_grad()

        output_real = discriminator(real_images)
        loss_real = loss_fn(output_real, real_labels)

        z = torch.randn(current_batch_size, latent_dim).to(device)
        fake_images = generator(z)

        output_fake = discriminator(fake_images.detach())
        loss_fake = loss_fn(output_fake, fake_labels)

        loss_d = loss_real + loss_fake

        loss_d.backward()
        optimizer_d.step()

        # ----------------------------
        # Train Generator
        # ----------------------------
        optimizer_g.zero_grad()

        output = discriminator(fake_images)
        loss_g = loss_fn(output, real_labels)

        loss_g.backward()
        optimizer_g.step()

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Loss D: {loss_d.item():.4f} "
        f"Loss G: {loss_g.item():.4f}"
    )

    # Save the same generated images every 10 epochs so the
    # Generator's improvement can be compared visually.
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            fake_images = generator(fixed_noise).cpu()

            save_image(
                fake_images,
                os.path.join(
                    "generated_images",
                    f"fake_images_epoch_{epoch + 1}.png"
                ),
                nrow=4,
                normalize=True
            )

print("Training finished!")


Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 470kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.47MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.2MB/s]


Starting training...
Epoch [1/50] Loss D: 0.1927 Loss G: 4.1764
Epoch [2/50] Loss D: 0.2266 Loss G: 4.7826
Epoch [3/50] Loss D: 0.0230 Loss G: 5.3268
Epoch [4/50] Loss D: 0.7213 Loss G: 2.6763
Epoch [5/50] Loss D: 0.2069 Loss G: 2.0842
Epoch [6/50] Loss D: 0.4123 Loss G: 3.5925
Epoch [7/50] Loss D: 0.4243 Loss G: 2.0730
Epoch [8/50] Loss D: 0.4371 Loss G: 2.0735
Epoch [9/50] Loss D: 0.3828 Loss G: 2.8131
Epoch [10/50] Loss D: 0.2763 Loss G: 3.9719
Epoch [11/50] Loss D: 0.4126 Loss G: 3.1490
Epoch [12/50] Loss D: 0.7271 Loss G: 1.4281
Epoch [13/50] Loss D: 0.7542 Loss G: 2.7325
Epoch [14/50] Loss D: 1.1924 Loss G: 2.1925
Epoch [15/50] Loss D: 0.8307 Loss G: 1.3952
Epoch [16/50] Loss D: 0.9531 Loss G: 2.2977
Epoch [17/50] Loss D: 1.0550 Loss G: 3.0538
Epoch [18/50] Loss D: 0.9070 Loss G: 0.7646
Epoch [19/50] Loss D: 0.6961 Loss G: 1.4376
Epoch [20/50] Loss D: 0.7941 Loss G: 2.9147
Epoch [21/50] Loss D: 1.5999 Loss G: 0.8928
Epoch [22/50] Loss D: 0.8197 Loss G: 0.8283
Epoch [23/50] Loss D